This jupyter notebook provided overall guidance on how to use the `utils` and `dkatlas` for general uses and creating base graphs implementing propagations

In [61]:
import networkx as nx, numpy as np, pandas as pd
import os, pickle
import importlib


import utils_a3 as utils
import desikan_killiany_atlas as dkatlas

importlib.reload(utils)
importlib.reload(dkatlas)

<module 'desikan_killiany_atlas' from 'C:\\Users\\sranasin\\PycharmProjects\\NetworkModels\\Project1\\A3\\desikan_killiany_atlas.py'>

### Basic Graph Creation and Save as Base

In [62]:
DK = dkatlas.DKAtlasGraph(xml_path=os.path.join("resources", "masters33.graphml"), base_rename=True)
DK.graph.number_of_nodes()

83

In [53]:
## Assign all types of weights
DK.assign_edge_weight(weight_function="weight1")
DK.assign_edge_weight(weight_function="weight2")

## Assign other properties
DK.set_ngeom_geom_property(geom_max_fiber_length=100)

In [54]:
DK.graph.nodes[1]

{'dn_position_x': 34.0419230769,
 'dn_position_y': 82.1757692308,
 'dn_position_z': 31.7769230769,
 'dn_correspondence_id': 1,
 'dn_region': 'cortical',
 'dn_fsname': 'lateralorbitofrontal',
 'dn_name': 'rh.lateralorbitofrontal',
 'dn_hemisphere': 'right',
 'region_name': 'rh_lateralorbitofrontal'}

In [55]:
list(DK.graph.edges(data=True))[0:1]

[(1,
  2,
  {'number_of_fibers': 5.62910798122,
   'FA_mean': 0.178233936269,
   'fiber_length_std': 0.816500084943,
   'fiber_length_mean': 15.9575699282,
   'FA_std': 0.116142955546,
   'weight1': 0.35275471181061957,
   'weight2': 0.022105791382886954,
   'type': 'geometric'})]

### Save Graph with all properties

In [22]:
importlib.reload(utils)
# utils.export_graphml_with_namespace(DK.graph, os.path.join("resources", "DKATLAS33_all.graphml"))

GraphML exported to: C:\Users\sranasin\PycharmProjects\NetworkModels\resources\DKATLAS33_all.graphml


### Save Graph with BASIC properties

In [56]:
node_properties_to_keep = ['dn_correspondence_id', 'region_name']
edge_properties_to_keep = ['weight1', 'weight2', 'type']

node_properties_to_remove = list(DK.graph.nodes[1].keys() - node_properties_to_keep)
edge_properties_to_remove = list(list(DK.graph.edges(data=True))[0:1][0][2].keys() - edge_properties_to_keep)

In [57]:
for _, attrs in DK.graph.nodes(data=True):
    for key in node_properties_to_remove:
        attrs.pop(key, None)

for _, _, attrs in DK.graph.edges(data=True):
    for key in edge_properties_to_remove:
        attrs.pop(key, None)

In [58]:
DK.graph.nodes[1]

{'dn_correspondence_id': 1, 'region_name': 'rh_lateralorbitofrontal'}

In [59]:
list(DK.graph.edges(data=True))[0:1]

[(1,
  2,
  {'weight1': 0.35275471181061957,
   'weight2': 0.022105791382886954,
   'type': 'geometric'})]

In [60]:
# utils.export_graphml_with_namespace(DK.graph, os.path.join("resources", "base_data", "DKATLAS33_base.graphml"))

GraphML exported to: C:\Users\sranasin\PycharmProjects\NetworkModels\resources\DKATLAS33_base.graphml


### Load Base File {as in for propagation}

In [31]:
graph = nx.read_graphml(os.path.join(utils.BASE_DIR, "resources", "DKATLAS33_all.graphml"))
nx.relabel_nodes(graph, lambda x: int(x), copy=False)

In [32]:
graph.nodes[1]

{'dn_position_x': 34.0419230769,
 'dn_position_y': 82.1757692308,
 'dn_position_z': 31.7769230769,
 'dn_correspondence_id': 1,
 'dn_region': 'cortical',
 'dn_fsname': 'lateralorbitofrontal',
 'dn_name': 'rh.lateralorbitofrontal',
 'dn_hemisphere': 'right',
 'region_name': 'rh_lateralorbitofrontal'}

In [33]:
list(graph.edges(data=True))[0:1]

[(1,
  2,
  {'number_of_fibers': 5.62910798122,
   'FA_mean': 0.178233936269,
   'fiber_length_std': 0.816500084943,
   'fiber_length_mean': 15.9575699282,
   'FA_std': 0.116142955546,
   'weight1': 0.35275471181061957,
   'weight2': 0.022105791382886954,
   'type': 'geometric'})]

### ADNI Amyloid Save Files (Activations, Snapshots, State Values)

For the setup a cleaned Dataset is selected.
* Dataset Filtered through `utils.safe_filter(base=True)`.
* Removes qc_flag <= 0, Tracers other than `FBP` and `FBB`, duplicate entries of `loniuid`

The problems with the Current Dataset
* For cross sectional study of cortical regions (within patient) SUVR values are appropriate.
* But for longitudinal study (of the same patient) `centiloids` must be used. But `centiloids` are only provided as a summary measure.
* Also for longitudinal study, you have to use a `composite_REF` rather than `whole_cerebellum_ref`. We must divide `cortical region SUVR/composite_REF_SUVR` for respective regions.  - But Problem (not sure)
* For Graph activations (amyloid positivity)  thresholds given per tracer is only recommended for summary values only. Hence cortical regional analysis is not possible.
* For analysis/comparison of patients with difference tracers - Not possible at the moment.

Activations: Applies Base Thresholding (Ignoring the warning of thresholding against cortical summary regions), (Currently ignoring composite REF Adjustments)
Snapshots: Some patients have regressed amyloid levels. (Meaning SUVR reduced, hence amyloid positivity returned back to -ive. This is ignored). Union(past, now) is considered to mitigate this

In [ ]:
# importlib.reload(utils)
# df_path = "resources/TEST_NEW/structured_files_UCBERKELEY_AMY_6MM_29Oct2025/UCBERKELEY_AMY_6MM_29Oct2025_suvr.csv"
# df = utils.df_rename_to_fsnames(df_path)
# df = utils.safe_filter_df(df, True)
# df.sort_values(by = ['rid', 'scandate'], inplace = True)
# df = df.head(4)
# df, feature_cols = utils.activations_cortical_regions_df(df, True)
# activation_times, snapshots, state_values = utils.activation_times_of_patients_for_cortical_regions_df(df, feature_cols, True)

Ran once to store the Data

`importlib.reload(utils)`

`df_path = "resources/TEST_NEW/structured_files_UCBERKELEY_AMY_6MM_29Oct2025/UCBERKELEY_AMY_6MM_29Oct2025_suvr.csv"`
`df = utils.df_rename_to_fsnames(df_path)`
`df = utils.safe_filter_df(df, filter_all=True)`
`df, feature_cols = utils.activations_cortical_regions_df(df, base_setup=True)`
`activations, snapshots, state_values = utils.activation_times_of_patients_for_cortical_regions_df(df,
                                                                                                  feature_cols=feature_cols, base_setup=True, save_files=False, save_files_path=os.path.join("resources", "base_data")
                                                                                                  )`


Can easily be called with

`activations, snapshots, state_values = utils._pull_saved_patient_data_files(path, path, path)`